# 04 — Risk-zone Evaluation & SHAP Analysis

Đánh giá theo vùng nguy hiểm và phân tích feature importance.

**Mục tiêu:**
1. Phân loại Normal/Warning/Exceedance
2. Đánh giá accuracy, precision, recall, F1 cho dangerous class
3. Hiểu vấn đề imbalance và Wilson CI
4. Phân tích feature importance theo horizon

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)

## 1. Chuẩn bị dữ liệu và huấn luyện

In [ ]:
from src.data.loader import load_raw_data, create_temporal_split, prepare_targets
from src.features.engineering import build_all_features, select_features_by_correlation
from src.models.train import train_model

df = load_raw_data()
df = prepare_targets(df, horizons=[1, 3, 7])
df = build_all_features(df).dropna()

splits = create_temporal_split(df)
train, val, test = splits['train'], splits['val'], splits['test']

target_cols = [c for c in df.columns if c.startswith('water_level_t')]
raw_cols = ['river_flow_a', 'river_flow_b', 'sea_level', 'water_level']
exclude = target_cols + raw_cols + ['month', 'day_of_year']
all_feature_cols = [c for c in df.columns if c not in exclude]

In [ ]:
# Huan luyen XGBoost cho moi horizon (gom ca train + val)
import xgboost as xgb

trainval = pd.concat([train, val])
predictions = {}

for horizon in [1, 3, 7]:
    target_col = f'water_level_t{horizon}'
    selected = select_features_by_correlation(trainval[all_feature_cols], trainval[target_col])
    
    X_trainval = trainval[selected].values
    y_trainval = trainval[target_col].values
    X_test = test[selected].values
    y_test = test[target_col].values
    
    model = xgb.XGBRegressor(n_estimators=500, max_depth=8, learning_rate=0.05,
                              subsample=0.8, random_state=42, verbosity=0, n_jobs=-1)
    model.fit(X_trainval, y_trainval)
    y_pred = model.predict(X_test)
    
    predictions[horizon] = {'y_true': y_test, 'y_pred': y_pred, 'model': model, 'features': selected}
    print(f't+{horizon}: {len(selected)} features, test n={len(y_test)}')

## 2. Risk-zone Evaluation

In [ ]:
from src.evaluation.risk_zone import classify_risk_zones, compute_risk_zone_metrics

# Nguong: P95=1.06m (Warning), P99=1.24m (Exceedance)
WARNING_TH = 1.06
EXCEEDANCE_TH = 1.24

print('='*80)
print('RISK-ZONE EVALUATION (XGBoost, nguong P95=1.06m, P99=1.24m)')
print('='*80)

for horizon in [1, 3, 7]:
    y_true = predictions[horizon]['y_true']
    y_pred = predictions[horizon]['y_pred']
    
    metrics = compute_risk_zone_metrics(y_true, y_pred, WARNING_TH, EXCEEDANCE_TH)
    
    print(f'\nt+{horizon}:')
    print(f'  So mau: {metrics["zone_counts"]}')
    print(f'  Accuracy 3-zone: {metrics["zone_accuracy"]:.4f}')
    print(f'  Precision (dangerous): {metrics["precision_dangerous"]:.3f}')
    print(f'  Recall (dangerous): {metrics["recall_dangerous"]:.3f}')
    print(f'  F1 (dangerous): {metrics["f1_dangerous"]:.3f}')
    print(f'  Exceedance recall: {metrics["exceedance_recall"]:.3f}')
    print(f'  Wilson 95% CI: [{metrics["wilson_ci"][0]:.3f}, {metrics["wilson_ci"][1]:.3f}]')
    print(f'  MAE theo vung: Normal={metrics["zone_mae"]["Normal"]:.4f}, '
          f'Warning={metrics["zone_mae"]["Warning"]:.4f}, '
          f'Exceedance={metrics["zone_mae"]["Exceedance"]:.4f}')

In [ ]:
# Truc quan hoa phan bo vung
from src.utils.plotting import plot_risk_zone_summary

metrics_t1 = compute_risk_zone_metrics(
    predictions[1]['y_true'], predictions[1]['y_pred'], WARNING_TH, EXCEEDANCE_TH
)
plot_risk_zone_summary(metrics_t1)

## 3. Phân tích Imbalance

**Vấn đề:** Test set có 684 Normal / 29 Warning / 9 Exceedance.

- 9 ngày Exceedance → Wilson CI [0.35, 0.88] → quá rộng
- Accuracy 99.5% chủ yếu do model dự đoán đúng Normal
- Cần thêm dữ liệu extreme events để kết luận mạnh

In [ ]:
# Tinh bias theo vung (under-predict hay over-predict?)
for horizon in [1, 3, 7]:
    y_true = predictions[horizon]['y_true']
    y_pred = predictions[horizon]['y_pred']
    
    zones = classify_risk_zones(y_true, WARNING_TH, EXCEEDANCE_TH)
    
    print(f'\nt+{horizon} — Bias (predicted - true) theo vung:')
    for zone in ['Normal', 'Warning', 'Exceedance']:
        mask = zones == zone
        if mask.sum() > 0:
            bias = np.mean(y_pred[mask] - y_true[mask])
            print(f'  {zone:12s}: bias = {bias:+.4f} m ({"under-predict" if bias < 0 else "over-predict"})')

print('\nLuu y: Bias am (under-predict) o ngay Exceedance → huong KHONG an toan cho canh bao lu!')

## 4. Feature Importance theo Horizon

In [ ]:
from src.utils.plotting import plot_feature_importance

fig, axes = plt.subplots(1, 3, figsize=(18, 8))

for ax, horizon in zip(axes, [1, 3, 7]):
    model = predictions[horizon]['model']
    features = predictions[horizon]['features']
    importances = model.feature_importances_
    
    indices = np.argsort(importances)[::-1][:10]
    ax.barh(range(10), importances[indices][::-1])
    ax.set_yticks(range(10))
    ax.set_yticklabels([features[i] for i in indices][::-1])
    ax.set_xlabel('Importance')
    ax.set_title(f't+{horizon}')
    ax.grid(axis='x', alpha=0.3)

plt.suptitle('Feature Importance theo Horizon (XGBoost)', fontsize=14)
plt.tight_layout()
plt.show()

print('\nNhan xet: Feature quan trọng thay doi theo horizon:')
print('  t+1: lag features (chu ky trieu)')
print('  t+3: is_rising (trạng thái tăng/giảm)')
print('  t+7: acceleration (gia tốc thay đổi)')

## Tóm tắt

**Kết quả chính:**
1. Risk-zone accuracy cao nhưng imbalance nghiêm trọng (9/722 Exceedance)
2. Wilson CI [0.35, 0.88] → không thể phân biệt với ngẫu nhiên
3. Bias âm ở Exceedance → under-predict → hướng không an toàn
4. Feature importance thay đổi theo horizon → cơ chế thủy văn khác nhau
5. Cần dữ liệu đo thực để validate hệ thống